# M06.P02 — De documentos a retriever

Código completo y ejecutable. El objetivo es ejecutar, inspeccionar, modificar y explicar.

Para asegurar una ejecución estable en CPU, este notebook usa representación TF-IDF local con scikit-learn. El objetivo es comprender el pipeline documento → chunk → vector → retrieval y la importancia de la metadata.


In [ ]:
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

class TfidfStore:
    def __init__(self, documents):
        self.documents = list(documents)
        self.vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([d.page_content for d in self.documents])

    def similarity_search(self, query, k=4):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix)[0]
        ranking = np.argsort(scores)[::-1][:k]
        return [self.documents[i] for i in ranking]

def parse_markdown(path: Path) -> Document:
    text = path.read_text(encoding="utf-8")
    metadata = {"file": path.name}
    body = text
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) == 3:
            raw_meta = parts[1]
            body = parts[2].strip()
            for line in raw_meta.splitlines():
                if ":" in line:
                    key, value = line.split(":", 1)
                    metadata[key.strip()] = value.strip().strip('\"')
    return Document(page_content=body, metadata=metadata)

def build_store(kb_path: Path):
    documents = [parse_markdown(p) for p in sorted(kb_path.glob("*.md"))]
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
    chunks = splitter.split_documents(documents)
    store = TfidfStore(chunks)
    return documents, chunks, store

def only_current(docs):
    return [d for d in docs if d.metadata.get("status") == "CURRENT"]

KB = Path("../assets/knowledge_base")
if not KB.exists():
    KB = Path("assets/knowledge_base")
documents, chunks, store = build_store(KB)

print("DOCUMENTOS")
for doc in documents:
    print(doc.metadata.get("source_id"), doc.metadata.get("version"), doc.metadata.get("status"))
print("\nCHUNKS:", len(chunks))

queries = ["¿Cuánto dura un acceso privilegiado?", "¿Cuándo se envían actualizaciones de un P1?"]
for query in queries:
    print("\nQUERY:", query)
    docs = store.similarity_search(query, k=4)
    for doc in docs:
        print(doc.metadata.get("source_id"), doc.metadata.get("version"), doc.metadata.get("status"), doc.page_content[:120].replace("\n", " "))
    safe_docs = only_current(docs)
    assert all(d.metadata.get("status") == "CURRENT" for d in safe_docs)
    print("CURRENT:", [(d.metadata.get("source_id"), d.metadata.get("version")) for d in safe_docs])


## Experimentos

1. Localiza la versión `CURRENT` y la `OBSOLETE` de `PROC-017`.
2. Cambia una consulta manteniendo el significado.
3. Observa si cambia el ranking.
4. Comprueba que `only_current()` impide utilizar documentos obsoletos.
